# Reproducible translation fine-tuning pipeline (Colab / Kaggle / local)

**Goal:** produce versioned Mandarin translation candidates with a fully configurable, resumable pipeline.

| Requirement | Where |
|---|---|
| Model, dataset, direction, method, seed, training settings, paths configurable | `configs/*.json` → `PipelineConfig` (section 3) |
| Method chosen after a hardware smoke test (LoRA / QLoRA / full) | `tlpipe/hardware.py`, section 4 |
| Data validated, versioned, hashed | `tlpipe/data.py` → `data_manifest.json` |
| Fine-tune / judge-calibration / prompt-validation / **sealed** final-test kept separate | four disjoint JSONL splits; `final_test` is read-only under `sealed/` and refused by `load_split()` |
| Dry runs, logging, checkpoints, resume | `dry_run=True` (section 5), `logs/`, `checkpoints/`, auto-resume (section 6) |
| Checkpoint loads and generates Mandarin | section 7 (fresh load + generation + Han-script check) |
| `candidate_manifest.json` with config, data version, hashes | section 8 |
| Tested code, configs, manifests, logs, report, notebook, GitHub link, loadable candidate | sections 2, 8, 9 |

**Runtime:** Colab *T4* (free) is enough for the default `Qwen/Qwen2.5-0.5B-Instruct` + 8k rows of OPUS-100 en→zh
(≈15–25 min). On Kaggle pick a GPU accelerator; paths are detected automatically.

## 1. Environment

In [1]:
%pip install -q "transformers>=4.44" "peft>=0.12" "datasets>=2.19" "accelerate>=0.30" "bitsandbytes>=0.43" "sacrebleu>=2.4" psutil
import torch, transformers, peft, datasets, sacrebleu
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 4.6 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.16.1 | peft 0.20.0 | cuda True Tesla T4


In [2]:
# Optional: persist runs to Google Drive on Colab (checkpoints survive disconnects -> resume works across sessions).
USE_DRIVE = False
import os
if USE_DRIVE and os.path.isdir("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TLPIPE_ROOT"] = "/content/drive/MyDrive/tlpipe_runs"
print("TLPIPE_ROOT =", os.environ.get("TLPIPE_ROOT", "(auto: Kaggle/Colab/local default)"))

TLPIPE_ROOT = (auto: Kaggle/Colab/local default)


## 2. Source code (tested modules)
The cells below write the package to disk exactly as it lives in the GitHub repo. Then the unit tests run.

In [3]:
!mkdir -p tlpipe tests configs && touch tlpipe/__init__.py

In [4]:
%%writefile tlpipe/config.py
"""Pipeline configuration: fully configurable, JSON round-trippable, hashable."""
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import asdict, dataclass, field, fields, is_dataclass
from typing import Any, Dict, List, Optional, get_type_hints

LANG_NAMES = {
    "en": "English", "zh": "Mandarin Chinese", "de": "German", "fr": "French",
    "es": "Spanish", "ja": "Japanese", "ko": "Korean", "ru": "Russian", "ar": "Arabic",
}


def runtime_root() -> str:
    """Pick a sensible run root for Kaggle, Colab or local."""
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/tlpipe_runs"
    if os.path.isdir("/content"):
        return "/content/tlpipe_runs"
    return os.path.abspath("./tlpipe_runs")


@dataclass
class DatasetConfig:
    source: str = "hf"                      # "hf" (datasets hub) or "local" (jsonl/csv/tsv)
    name: str = "Helsinki-NLP/opus-100"     # hf dataset id
    config: Optional[str] = "en-zh"         # hf dataset config
    split: str = "train"                    # hf split to draw from
    path: Optional[str] = None              # local file for source="local"
    translation_column: Optional[str] = "translation"  # nested column ({"en":..,"zh":..}); None = flat columns
    src_field: str = "en"
    tgt_field: str = "zh"
    max_rows: int = 8000                    # cap (after validation, before splitting)
    min_chars: int = 1
    max_chars: int = 400
    han_ratio_min: float = 0.3              # required Han-character ratio on the Mandarin side
    split_ratios: Dict[str, float] = field(default_factory=lambda: {
        "finetune": 0.88, "judge_calibration": 0.04, "prompt_validation": 0.04, "final_test": 0.04})
    dev_fraction: float = 0.05              # carved out of `finetune` for loss monitoring only


@dataclass
class TrainConfig:
    method: str = "auto"                    # auto | lora | qlora | full
    epochs: float = 1.0
    max_steps: int = -1                     # >0 overrides epochs
    learning_rate: float = 2e-4
    per_device_batch_size: int = 4
    grad_accum: int = 4
    max_len: int = 256
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    logging_steps: int = 10
    save_steps: int = 100
    save_total_limit: int = 3
    eval_steps: int = 100
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    gradient_checkpointing: bool = True
    resume: bool = True                     # resume from last checkpoint in output_dir if present
    dry_run_steps: int = 3
    dry_run_rows: int = 32


@dataclass
class EvalConfig:
    n_prompt_validation: int = 50
    n_judge_calibration: int = 30
    max_new_tokens: int = 128
    save_merged: bool = False               # additionally save merged full weights


@dataclass
class PipelineConfig:
    run_name: str = "mandarin-candidate"
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    direction: str = "en-zh"                # "<src>-<tgt>"
    seed: int = 42
    dry_run: bool = False
    root_dir: Optional[str] = None            # None -> auto (Kaggle/Colab/local)
    output_dir: Optional[str] = None
    data_dir: Optional[str] = None
    log_dir: Optional[str] = None
    repo_url: str = "https://github.com/<org>/<repo>"
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)

    def __post_init__(self):
        if self.root_dir is None:
            self.root_dir = runtime_root()
        if self.output_dir is None:
            self.output_dir = os.path.join(self.root_dir, self.run_name)
        if self.data_dir is None:
            self.data_dir = os.path.join(self.root_dir, "data", self.direction)
        if self.log_dir is None:
            self.log_dir = os.path.join(self.output_dir, "logs")
        if "-" not in self.direction:
            raise ValueError("direction must look like 'en-zh'")

    # ---- language helpers -------------------------------------------------
    @property
    def src_lang(self) -> str:
        return self.direction.split("-")[0]

    @property
    def tgt_lang(self) -> str:
        return self.direction.split("-")[1]

    def lang_name(self, code: str) -> str:
        return LANG_NAMES.get(code, code)

    # ---- serialisation ----------------------------------------------------
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def to_json(self, path: str) -> str:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=2, ensure_ascii=False)
        return path

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "PipelineConfig":
        return _build(cls, d)

    @classmethod
    def from_json(cls, path: str) -> "PipelineConfig":
        with open(path, encoding="utf-8") as f:
            return cls.from_dict(json.load(f))

    def config_hash(self) -> str:
        """Stable hash of everything that influences training (paths excluded)."""
        d = self.to_dict()
        for k in ("root_dir", "output_dir", "data_dir", "log_dir", "repo_url"):
            d.pop(k, None)
        blob = json.dumps(d, sort_keys=True, ensure_ascii=False).encode("utf-8")
        return hashlib.sha256(blob).hexdigest()


def _build(cls, d: Dict[str, Any]):
    hints = get_type_hints(cls)
    kwargs = {}
    for f in fields(cls):
        if f.name not in d:
            continue
        v = d[f.name]
        t = hints.get(f.name)
        if isinstance(t, type) and is_dataclass(t) and isinstance(v, dict):
            v = _build(t, v)
        kwargs[f.name] = v
    return cls(**kwargs)

Writing tlpipe/config.py


In [5]:
%%writefile tlpipe/utils.py
"""Small shared helpers: logging, seeding, hashing, JSONL IO, environment capture."""
from __future__ import annotations

import datetime as _dt
import hashlib
import json
import logging
import os
import platform
import random
import subprocess
import sys
from typing import Any, Dict, Iterable, List, Optional


def utc_now() -> str:
    return _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def setup_logging(log_dir: str, name: str = "tlpipe") -> logging.Logger:
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    path = os.path.join(log_dir, "pipeline.log")
    if not any(getattr(h, "_tlpipe_path", None) == path for h in logger.handlers):
        fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
        fh = logging.FileHandler(path, encoding="utf-8")
        fh.setFormatter(fmt)
        fh._tlpipe_path = path  # type: ignore[attr-defined]
        logger.addHandler(fh)
        if not any(isinstance(h, logging.StreamHandler) and not isinstance(h, logging.FileHandler)
                   for h in logger.handlers):
            sh = logging.StreamHandler(sys.stdout)
            sh.setFormatter(fmt)
            logger.addHandler(sh)
    return logger


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def sha256_file(path: str, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_dir(path: str, patterns: Iterable[str] = (".safetensors", ".bin", ".json")) -> Dict[str, str]:
    out = {}
    for root, _, files in os.walk(path):
        for fn in sorted(files):
            if any(fn.endswith(p) for p in patterns):
                fp = os.path.join(root, fn)
                out[os.path.relpath(fp, path)] = sha256_file(fp)
    return out


def write_jsonl(path: str, rows: Iterable[Dict[str, Any]]) -> int:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            n += 1
    return n


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_json(path: str, obj: Any) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    return path


def read_json(path: str) -> Any:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def git_info(cwd: Optional[str] = None) -> Dict[str, Optional[str]]:
    def _run(args):
        try:
            return subprocess.check_output(["git"] + args, cwd=cwd, stderr=subprocess.DEVNULL,
                                           text=True).strip()
        except Exception:
            return None
    return {"commit": _run(["rev-parse", "HEAD"]), "branch": _run(["rev-parse", "--abbrev-ref", "HEAD"]),
            "remote": _run(["config", "--get", "remote.origin.url"]),
            "dirty": (lambda s: None if s is None else bool(s))(_run(["status", "--porcelain"]))}


def env_info() -> Dict[str, Any]:
    info: Dict[str, Any] = {"python": sys.version.split()[0], "platform": platform.platform(),
                            "timestamp_utc": utc_now(), "packages": {}}
    for pkg in ("torch", "transformers", "peft", "datasets", "bitsandbytes", "accelerate", "sacrebleu"):
        try:
            mod = __import__(pkg)
            info["packages"][pkg] = getattr(mod, "__version__", "unknown")
        except Exception:
            info["packages"][pkg] = None
    return info

Writing tlpipe/utils.py


In [6]:
%%writefile tlpipe/hardware.py
"""Hardware smoke test + feasible-method selection (LoRA / QLoRA / full)."""
from __future__ import annotations

import os
import shutil
from typing import Any, Dict, Tuple

METHODS = ("lora", "qlora", "full")


def detect_platform() -> str:
    if os.path.isdir("/kaggle/working"):
        return "kaggle"
    if os.path.isdir("/content"):
        return "colab"
    return "local"


def smoke_test(logger=None) -> Dict[str, Any]:
    """Probe the machine: GPU, VRAM, bf16, bitsandbytes, a tiny matmul, disk and RAM."""
    info: Dict[str, Any] = {
        "platform": detect_platform(), "cuda": False, "gpu": None, "vram_gb": 0.0,
        "bf16": False, "bitsandbytes": False, "matmul_ok": False, "torch": None,
        "free_disk_gb": round(shutil.disk_usage(".").free / 1e9, 1), "ram_gb": None, "errors": [],
    }
    try:
        import psutil  # type: ignore
        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.is_available()
        if info["cuda"]:
            p = torch.cuda.get_device_properties(0)
            info["gpu"] = p.name
            info["vram_gb"] = round(p.total_memory / 1e9, 1)
            try:
                info["bf16"] = bool(torch.cuda.is_bf16_supported())
            except Exception:
                info["bf16"] = False
        dev = "cuda" if info["cuda"] else "cpu"
        a = torch.randn(256, 256, device=dev)
        info["matmul_ok"] = bool(torch.isfinite((a @ a).sum()).item())
    except Exception as e:  # pragma: no cover
        info["errors"].append(f"torch: {e!r}")
    try:
        import bitsandbytes as bnb  # noqa: F401
        info["bitsandbytes"] = bool(info["cuda"])  # 4-bit needs CUDA
        if info["cuda"]:
            import torch
            lin = bnb.nn.Linear4bit(64, 64, compute_dtype=torch.float16).cuda()
            lin(torch.randn(2, 64, device="cuda", dtype=torch.float16))
    except Exception as e:
        info["bitsandbytes"] = False
        info["errors"].append(f"bitsandbytes: {e!r}")
    if logger:
        logger.info("smoke_test: %s", info)
    return info


def select_method(info: Dict[str, Any], requested: str = "auto") -> Tuple[str, str]:
    """Return (method, reason). Honours an explicit request when feasible, otherwise downgrades."""
    cuda, vram, bnb = info.get("cuda"), float(info.get("vram_gb") or 0), info.get("bitsandbytes")
    if not info.get("matmul_ok", True):
        raise RuntimeError("Smoke test failed: basic matmul did not run; fix the environment first.")
    if requested == "auto":
        if not cuda:
            return "lora", "no GPU: LoRA on CPU (only suitable for dry runs / tiny models)"
        if vram >= 20:
            return "lora", f"GPU with {vram} GB VRAM: standard 16-bit LoRA fits comfortably"
        if bnb:
            return "qlora", f"GPU with {vram} GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen"
        return "lora", f"GPU with {vram} GB VRAM, bitsandbytes unavailable: LoRA"
    if requested not in METHODS:
        raise ValueError(f"unknown method {requested!r}; choose auto|lora|qlora|full")
    if requested == "qlora" and not bnb:
        return "lora", "QLoRA requested but bitsandbytes/CUDA unavailable: downgraded to LoRA"
    if requested == "full" and vram < 40:
        return ("qlora" if bnb else "lora"), f"full fine-tune requested but only {vram} GB VRAM: downgraded"
    return requested, f"{requested} requested and feasible on this hardware"

Writing tlpipe/hardware.py


In [7]:
%%writefile tlpipe/data.py
"""Data loading, validation, deterministic splitting, versioning, hashing and sealing.

Splits produced (all disjoint by source sentence):
  finetune            -> training (a `dev` slice is carved out for loss monitoring only)
  judge_calibration   -> outputs for calibrating an LLM/human judge; never trained on
  prompt_validation   -> generation-based validation while iterating on prompts
  final_test (SEALED) -> stored under data_dir/sealed/, hash recorded, refused by
                         load_split() unless unseal=True. Evaluate once, at the very end.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
from collections import Counter
from typing import Any, Dict, Iterable, List, Optional, Tuple

from .config import DatasetConfig, PipelineConfig
from .utils import read_json, read_jsonl, sha256_file, utc_now, write_json, write_jsonl

HAN_RE = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]")
SPLIT_NAMES = ("finetune", "judge_calibration", "prompt_validation", "final_test")
SEALED_SPLITS = ("final_test",)


class SealedSplitError(PermissionError):
    pass


def han_ratio(text: str) -> float:
    letters = [c for c in text if not c.isspace()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if HAN_RE.match(c)) / len(letters)


# --------------------------------------------------------------------------- load
def load_raw_rows(ds: DatasetConfig, limit: Optional[int] = None) -> List[Dict[str, str]]:
    """Return a list of {"src":..., "tgt":...} from the HF hub or a local file."""
    rows: List[Dict[str, str]] = []
    if ds.source == "local":
        if not ds.path or not os.path.exists(ds.path):
            raise FileNotFoundError(f"local dataset not found: {ds.path}")
        if ds.path.endswith(".jsonl"):
            raw = read_jsonl(ds.path)
        elif ds.path.endswith((".csv", ".tsv")):
            with open(ds.path, encoding="utf-8", newline="") as f:
                raw = list(csv.DictReader(f, delimiter="\t" if ds.path.endswith(".tsv") else ","))
        else:
            raise ValueError("local dataset must be .jsonl, .csv or .tsv")
        for r in raw:
            src, tgt = _pick(r, ds)
            rows.append({"src": src, "tgt": tgt})
            if limit and len(rows) >= limit:
                break
    elif ds.source == "hf":
        from datasets import load_dataset  # lazy import
        split = ds.split if not limit else f"{ds.split}[:{limit}]"
        hf = load_dataset(ds.name, ds.config, split=split)
        for r in hf:
            src, tgt = _pick(r, ds)
            rows.append({"src": src, "tgt": tgt})
    else:
        raise ValueError(f"unknown dataset source {ds.source!r}")
    return rows


def _pick(r: Dict[str, Any], ds: DatasetConfig) -> Tuple[Any, Any]:
    if ds.translation_column and isinstance(r.get(ds.translation_column), dict):
        inner = r[ds.translation_column]
        return inner.get(ds.src_field), inner.get(ds.tgt_field)
    return r.get(ds.src_field), r.get(ds.tgt_field)


# ----------------------------------------------------------------------- validate
def validate_rows(rows: Iterable[Dict[str, Any]], ds: DatasetConfig, direction: str
                  ) -> Tuple[List[Dict[str, str]], Dict[str, int]]:
    """Filter rows. Returns (clean_rows, stats). Deterministic, order-preserving."""
    src_lang, tgt_lang = direction.split("-")
    stats: Counter = Counter()
    seen_src = set()
    clean: List[Dict[str, str]] = []
    for r in rows:
        stats["total"] += 1
        src, tgt = r.get("src"), r.get("tgt")
        if not isinstance(src, str) or not isinstance(tgt, str):
            stats["drop_not_string"] += 1
            continue
        src, tgt = src.strip(), tgt.strip()
        if not src or not tgt:
            stats["drop_empty"] += 1
            continue
        if not (ds.min_chars <= len(src) <= ds.max_chars and ds.min_chars <= len(tgt) <= ds.max_chars):
            stats["drop_length"] += 1
            continue
        if src == tgt:
            stats["drop_identical"] += 1
            continue
        if tgt_lang == "zh" and han_ratio(tgt) < ds.han_ratio_min:
            stats["drop_target_not_mandarin"] += 1
            continue
        if src_lang == "zh" and han_ratio(src) < ds.han_ratio_min:
            stats["drop_source_not_mandarin"] += 1
            continue
        if src_lang != "zh" and han_ratio(src) > 0.5:
            stats["drop_source_looks_mandarin"] += 1
            continue
        if src in seen_src:
            stats["drop_duplicate_source"] += 1
            continue
        seen_src.add(src)
        clean.append({"src": src, "tgt": tgt})
        stats["kept"] += 1
    return clean, dict(stats)


# -------------------------------------------------------------------------- split
def row_hash(r: Dict[str, str]) -> str:
    return hashlib.sha256(json.dumps([r["src"], r["tgt"]], ensure_ascii=False).encode("utf-8")).hexdigest()


def split_rows(rows: List[Dict[str, str]], ratios: Dict[str, float], seed: int,
               max_rows: Optional[int] = None) -> Dict[str, List[Dict[str, str]]]:
    """Seeded shuffle then contiguous slices. Sources are unique -> splits are disjoint."""
    for name in SPLIT_NAMES:
        if name not in ratios:
            raise ValueError(f"split_ratios must contain {name}")
    if abs(sum(ratios.values()) - 1.0) > 1e-6:
        raise ValueError("split_ratios must sum to 1.0")
    idx = list(range(len(rows)))
    random.Random(seed).shuffle(idx)
    if max_rows:
        idx = idx[:max_rows]
    n = len(idx)
    out: Dict[str, List[Dict[str, str]]] = {}
    start = 0
    for i, name in enumerate(SPLIT_NAMES):
        end = n if i == len(SPLIT_NAMES) - 1 else start + int(round(n * ratios[name]))
        out[name] = [dict(rows[j], id=row_hash(rows[j])[:16]) for j in idx[start:end]]
        start = end
    return out


def compute_data_version(splits: Dict[str, List[Dict[str, str]]], ds: DatasetConfig, seed: int) -> str:
    h = hashlib.sha256()
    for name in SPLIT_NAMES:
        h.update(name.encode())
        for r in splits[name]:
            h.update(row_hash(r).encode())
    h.update(json.dumps({"validation": {"min_chars": ds.min_chars, "max_chars": ds.max_chars,
                                        "han_ratio_min": ds.han_ratio_min}, "seed": seed},
                        sort_keys=True).encode())
    return "v-" + h.hexdigest()[:12]


# ------------------------------------------------------------------ prepare/verify
def split_path(data_dir: str, name: str) -> str:
    if name in SEALED_SPLITS:
        return os.path.join(data_dir, "sealed", f"{name}.jsonl")
    return os.path.join(data_dir, f"{name}.jsonl")


def prepare_data(cfg: PipelineConfig, logger=None, force: bool = False) -> Dict[str, Any]:
    """Load -> validate -> split -> write -> hash -> data_manifest.json. Idempotent."""
    ds, data_dir = cfg.dataset, cfg.data_dir
    manifest_path = os.path.join(data_dir, "data_manifest.json")
    if os.path.exists(manifest_path) and not force:
        ok, problems = verify_data(data_dir)
        old = read_json(manifest_path)
        if ok and old.get("source") == _source_desc(ds) and old.get("seed") == cfg.seed:
            if logger:
                logger.info("data: reusing verified %s at %s", old["data_version"], data_dir)
            return old
        if logger:
            logger.warning("data: existing manifest mismatch (%s); regenerating", problems or "config changed")
    limit = ds.max_rows * 2 if ds.max_rows else None  # over-fetch to survive validation drops
    raw = load_raw_rows(ds, limit=limit)
    clean, stats = validate_rows(raw, ds, cfg.direction)
    if len(clean) < 50:
        raise ValueError(f"only {len(clean)} valid rows after validation; check dataset/fields. stats={stats}")
    splits = split_rows(clean, ds.split_ratios, cfg.seed, ds.max_rows)
    for name in SPLIT_NAMES:
        if not splits[name]:
            raise ValueError(f"split {name} is empty; increase max_rows or adjust split_ratios")
    version = compute_data_version(splits, ds, cfg.seed)
    manifest: Dict[str, Any] = {"data_version": version, "created_utc": utc_now(), "direction": cfg.direction,
                                "seed": cfg.seed, "source": _source_desc(ds), "validation_stats": stats,
                                "splits": {}, "sealed": {}}
    for name in SPLIT_NAMES:
        p = split_path(data_dir, name)
        n = write_jsonl(p, splits[name])
        entry = {"path": p, "rows": n, "sha256": sha256_file(p)}
        if name in SEALED_SPLITS:
            manifest["sealed"][name] = entry
            os.chmod(p, 0o444)
            write_json(os.path.join(data_dir, "sealed", "SEALED.json"),
                       {"note": "Do not open until final evaluation. Verify sha256 before use.",
                        "created_utc": manifest["created_utc"], name: entry["sha256"]})
        else:
            manifest["splits"][name] = entry
    write_json(manifest_path, manifest)
    if logger:
        logger.info("data: %s prepared -> %s | stats=%s", version, data_dir, stats)
    return manifest


def _source_desc(ds: DatasetConfig) -> Dict[str, Any]:
    return {"source": ds.source, "name": ds.name, "config": ds.config, "split": ds.split, "path": ds.path,
            "src_field": ds.src_field, "tgt_field": ds.tgt_field, "max_rows": ds.max_rows}


def verify_data(data_dir: str) -> Tuple[bool, List[str]]:
    """Recompute hashes of every split (incl. sealed) and compare with data_manifest.json."""
    manifest_path = os.path.join(data_dir, "data_manifest.json")
    if not os.path.exists(manifest_path):
        return False, ["data_manifest.json missing"]
    m = read_json(manifest_path)
    problems = []
    for group in ("splits", "sealed"):
        for name, e in m.get(group, {}).items():
            if not os.path.exists(e["path"]):
                problems.append(f"{name}: file missing")
            elif sha256_file(e["path"]) != e["sha256"]:
                problems.append(f"{name}: sha256 mismatch")
    return not problems, problems


def load_split(data_dir: str, name: str, unseal: bool = False, n: Optional[int] = None
               ) -> List[Dict[str, str]]:
    if name in SEALED_SPLITS and not unseal:
        raise SealedSplitError(f"'{name}' is sealed. Pass unseal=True only for the one final evaluation.")
    rows = read_jsonl(split_path(data_dir, name))
    return rows[:n] if n else rows


def finetune_train_dev(data_dir: str, dev_fraction: float, seed: int, n: Optional[int] = None
                       ) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    rows = load_split(data_dir, "finetune")
    random.Random(seed).shuffle(rows)
    n_dev = max(1, int(len(rows) * dev_fraction))
    dev, train = rows[:n_dev], rows[n_dev:]
    if n:
        train, dev = train[:n], dev[: max(1, n // 4)]
    return train, dev

Writing tlpipe/data.py


In [8]:
%%writefile tlpipe/train.py
"""Fine-tuning: LoRA / QLoRA / full with checkpoints, resume, JSONL logging and dry runs."""
from __future__ import annotations

import inspect
import json
import math
import os
import time
from typing import Any, Dict, List, Optional, Tuple

from .config import PipelineConfig
from .data import finetune_train_dev, verify_data
from .utils import write_json


# ----------------------------------------------------------------------- prompting
def system_prompt(cfg: PipelineConfig) -> str:
    return (f"You are a professional translator. Translate the user's {cfg.lang_name(cfg.src_lang)} text "
            f"into {cfg.lang_name(cfg.tgt_lang)}. Output only the translation, nothing else.")


def build_messages(cfg: PipelineConfig, src: str) -> List[Dict[str, str]]:
    return [{"role": "system", "content": system_prompt(cfg)}, {"role": "user", "content": src}]


def render_prompt(tok, cfg: PipelineConfig, src: str) -> str:
    return tok.apply_chat_template(build_messages(cfg, src), tokenize=False, add_generation_prompt=True)


def encode_example(tok, cfg: PipelineConfig, src: str, tgt: str, max_len: int) -> Dict[str, List[int]]:
    """Prompt tokens are masked (-100) so loss is only on the translation."""
    prompt = render_prompt(tok, cfg, src)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    t_ids = tok(tgt + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p_ids + t_ids)[:max_len]
    labels = ([-100] * len(p_ids) + t_ids)[:max_len]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}


class PadCollator:
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, feats: List[Dict[str, List[int]]]) -> Dict[str, Any]:
        import torch
        m = max(len(f["input_ids"]) for f in feats)
        pad = lambda seq, v: seq + [v] * (m - len(seq))  # noqa: E731
        return {"input_ids": torch.tensor([pad(f["input_ids"], self.pad_id) for f in feats]),
                "attention_mask": torch.tensor([pad(f["attention_mask"], 0) for f in feats]),
                "labels": torch.tensor([pad(f["labels"], -100) for f in feats])}


# ------------------------------------------------------------------- model loading
def load_tokenizer(model_name: str):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok


def load_base_model(model_name: str, method: str, smoke: Dict[str, Any], for_training: bool = True):
    import torch
    from transformers import AutoModelForCausalLM
    import transformers
    cuda = smoke.get("cuda")
    dtype = torch.bfloat16 if smoke.get("bf16") else (torch.float16 if cuda else torch.float32)
    if method == "full" and for_training and dtype == torch.float16:
        dtype = torch.float32  # fp16 master weights cannot be trained by the AMP scaler
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    kw: Dict[str, Any] = {("dtype" if (major, minor) >= (4, 56) else "torch_dtype"): dtype}
    if method == "qlora":
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True,
                                                       bnb_4bit_compute_dtype=dtype)
        kw["device_map"] = {"": 0}
    elif cuda:
        kw["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(model_name, **kw)
    model.config.use_cache = not for_training
    return model, dtype


def wrap_peft(model, cfg: PipelineConfig, method: str):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if method == "full":
        return model
    if method == "qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.train.gradient_checkpointing)
    elif cfg.train.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    lcfg = LoraConfig(r=cfg.train.lora_r, lora_alpha=cfg.train.lora_alpha, lora_dropout=cfg.train.lora_dropout,
                      target_modules=cfg.train.target_modules, bias="none", task_type="CAUSAL_LM")
    return get_peft_model(model, lcfg)


# ------------------------------------------------------------------------- logging
def _jsonl_callback(path: str):
    from transformers import TrainerCallback

    class JsonlLogger(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kw):
            if logs:
                with open(path, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"step": state.global_step, "epoch": state.epoch, **logs}) + "\n")
    return JsonlLogger()


def find_last_checkpoint(output_dir: str) -> Optional[str]:
    try:
        from transformers.trainer_utils import get_last_checkpoint
        return get_last_checkpoint(output_dir) if os.path.isdir(output_dir) else None
    except Exception:
        return None


def _supported_params(cls) -> Tuple[set, bool]:
    """Parameter names accepted by cls.__init__, and whether it takes **kwargs."""
    params = inspect.signature(cls.__init__).parameters
    var_kw = any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())
    return set(params) - {"self"}, var_kw


def _estimate_total_steps(cfg: PipelineConfig, n_rows: Optional[int]) -> int:
    """Optimizer steps the run will take; used when only warmup_steps is supported."""
    max_steps = cfg.train.dry_run_steps if cfg.dry_run else cfg.train.max_steps
    if max_steps and max_steps > 0:
        return int(max_steps)
    if not n_rows:
        return 0
    per_epoch = math.ceil(n_rows / max(1, cfg.train.per_device_batch_size * cfg.train.grad_accum))
    return max(1, int(per_epoch * cfg.train.epochs))


def _training_args(cfg: PipelineConfig, method: str, smoke: Dict[str, Any], ckpt_dir: str,
                   n_rows: Optional[int] = None, logger=None):
    """Build TrainingArguments, tolerating renamed/removed arguments across transformers versions."""
    from transformers import TrainingArguments
    t, cuda = cfg.train, smoke.get("cuda")
    supported, var_kw = _supported_params(TrainingArguments)
    max_steps = t.dry_run_steps if cfg.dry_run else t.max_steps
    kw: Dict[str, Any] = dict(
        output_dir=ckpt_dir, seed=cfg.seed, data_seed=cfg.seed,
        per_device_train_batch_size=t.per_device_batch_size, per_device_eval_batch_size=t.per_device_batch_size,
        gradient_accumulation_steps=t.grad_accum, learning_rate=t.learning_rate,
        num_train_epochs=t.epochs, max_steps=max_steps,
        weight_decay=t.weight_decay, lr_scheduler_type="cosine",
        logging_steps=1 if cfg.dry_run else t.logging_steps, logging_first_step=True,
        save_steps=1 if cfg.dry_run else t.save_steps, save_total_limit=t.save_total_limit,
        eval_steps=1 if cfg.dry_run else t.eval_steps, save_strategy="steps",
        bf16=bool(cuda and smoke.get("bf16")), fp16=bool(cuda and not smoke.get("bf16")),
        report_to="none", remove_unused_columns=False, dataloader_num_workers=0,
        optim="paged_adamw_8bit" if method == "qlora" else "adamw_torch",
        gradient_checkpointing=False,  # handled on the model in wrap_peft
    )

    # eval strategy was renamed evaluation_strategy -> eval_strategy
    for alias in ("eval_strategy", "evaluation_strategy"):
        if var_kw or alias in supported:
            kw[alias] = "steps"
            break

    # warmup_ratio was removed in some builds; fall back to an equivalent warmup_steps
    if var_kw or "warmup_ratio" in supported:
        kw["warmup_ratio"] = t.warmup_ratio
    elif "warmup_steps" in supported:
        total = _estimate_total_steps(cfg, n_rows)
        kw["warmup_steps"] = max(0, int(round(total * t.warmup_ratio)))
        if logger:
            logger.info("TrainingArguments: warmup_ratio unsupported -> warmup_steps=%s (of ~%s total)",
                        kw["warmup_steps"], total)

    if not var_kw:
        dropped = sorted(k for k in kw if k not in supported)
        for k in dropped:
            kw.pop(k)
        if dropped:
            msg = ("TrainingArguments (transformers %s) does not accept %s; dropped and using its defaults."
                   % (__import__("transformers").__version__, dropped))
            (logger.warning if logger else print)(msg)
    return TrainingArguments(**kw)


# ----------------------------------------------------------------------------- run
def train(cfg: PipelineConfig, method: str, smoke: Dict[str, Any], logger=None) -> Dict[str, Any]:
    """Train and return a summary dict. Never touches the sealed split."""
    import torch
    from datasets import Dataset
    from transformers import Trainer

    ok, problems = verify_data(cfg.data_dir)
    if not ok:
        raise RuntimeError(f"data verification failed before training: {problems}")

    ckpt_dir = os.path.join(cfg.output_dir, "checkpoints")
    log_path = os.path.join(cfg.log_dir, "train_log.jsonl")
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(cfg.log_dir, exist_ok=True)

    n_rows = cfg.train.dry_run_rows if cfg.dry_run else None
    train_rows, dev_rows = finetune_train_dev(cfg.data_dir, cfg.dataset.dev_fraction, cfg.seed, n=n_rows)
    tok = load_tokenizer(cfg.model_name)
    enc = lambda r: encode_example(tok, cfg, r["src"], r["tgt"], cfg.train.max_len)  # noqa: E731
    train_ds = Dataset.from_list([enc(r) for r in train_rows])
    dev_ds = Dataset.from_list([enc(r) for r in dev_rows])
    if logger:
        logger.info("train: method=%s rows train=%d dev=%d dry_run=%s", method, len(train_ds), len(dev_ds), cfg.dry_run)

    model, dtype = load_base_model(cfg.model_name, method, smoke, for_training=True)
    model = wrap_peft(model, cfg, method)
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    args = _training_args(cfg, method, smoke, ckpt_dir, n_rows=len(train_ds), logger=logger)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=PadCollator(tok.pad_token_id), callbacks=[_jsonl_callback(log_path)])

    resume_from = find_last_checkpoint(ckpt_dir) if cfg.train.resume else None
    if logger:
        logger.info("train: resume_from=%s", resume_from)
    t0 = time.time()
    result = trainer.train(resume_from_checkpoint=resume_from)
    elapsed = time.time() - t0

    cand_dir = os.path.join(cfg.output_dir, "candidate")
    os.makedirs(cand_dir, exist_ok=True)
    trainer.save_model(cand_dir)            # adapter (or full weights for method=full)
    tok.save_pretrained(cand_dir)
    if cfg.eval.save_merged and method != "full":
        merged = model.merge_and_unload()
        merged.save_pretrained(os.path.join(cfg.output_dir, "candidate_merged"), safe_serialization=True)
        tok.save_pretrained(os.path.join(cfg.output_dir, "candidate_merged"))

    summary = {"method": method, "dtype": str(dtype), "resumed_from": resume_from,
               "global_step": trainer.state.global_step, "train_loss": result.metrics.get("train_loss"),
               "train_runtime_sec": round(elapsed, 1), "rows_train": len(train_ds), "rows_dev": len(dev_ds),
               "trainable_params": trainable, "total_params": total,
               "log_history": trainer.state.log_history[-5:], "candidate_dir": cand_dir,
               "checkpoint_dir": ckpt_dir, "last_checkpoint": find_last_checkpoint(ckpt_dir)}
    try:
        summary["eval_loss"] = trainer.evaluate().get("eval_loss")
    except Exception as e:  # pragma: no cover
        summary["eval_loss_error"] = repr(e)
    write_json(os.path.join(cfg.output_dir, "train_summary.json"), summary)
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

Writing tlpipe/train.py


In [9]:
%%writefile tlpipe/evaluate.py
"""Load the candidate checkpoint, generate translations, score, and (once) run the sealed test."""
from __future__ import annotations

import os
from typing import Any, Dict, List, Optional

from .config import PipelineConfig
from .data import han_ratio, load_split, split_path, verify_data
from .train import load_base_model, load_tokenizer, render_prompt
from .utils import read_json, sha256_file, utc_now, write_json, write_jsonl


def load_candidate(cfg: PipelineConfig, method: str, smoke: Dict[str, Any], candidate_dir: Optional[str] = None):
    """Fresh load of base + adapter (or full weights). Proves the checkpoint is loadable."""
    import torch
    cand = candidate_dir or os.path.join(cfg.output_dir, "candidate")
    if not os.path.isdir(cand):
        raise FileNotFoundError(f"candidate not found: {cand}")
    tok = load_tokenizer(cand)
    tok.padding_side = "left"
    if method == "full":
        model, _ = load_base_model(cand, "full", smoke, for_training=False)
    else:
        from peft import PeftModel
        base, _ = load_base_model(cfg.model_name, method, smoke, for_training=False)
        model = PeftModel.from_pretrained(base, cand)
    model.eval()
    return tok, model


def translate(tok, model, cfg: PipelineConfig, sources: List[str], batch_size: int = 8) -> List[str]:
    import torch
    outs: List[str] = []
    for i in range(0, len(sources), batch_size):
        batch = [render_prompt(tok, cfg, s) for s in sources[i:i + batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=cfg.eval.max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id, repetition_penalty=1.05)
        for j in range(gen.shape[0]):
            outs.append(tok.decode(gen[j, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs


def score(hyps: List[str], refs: List[str], tgt_lang: str) -> Dict[str, float]:
    import sacrebleu
    tokenize = "zh" if tgt_lang == "zh" else "13a"
    bleu = sacrebleu.corpus_bleu(hyps, [refs], tokenize=tokenize)
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    han = sum(1 for h in hyps if han_ratio(h) >= 0.3) / max(1, len(hyps))
    return {"bleu": round(bleu.score, 2), "chrf": round(chrf.score, 2), "n": len(hyps),
            "mandarin_output_rate": round(han, 3) if tgt_lang == "zh" else None}


def _run_set(tok, model, cfg, rows, out_path):
    hyps = translate(tok, model, cfg, [r["src"] for r in rows])
    write_jsonl(out_path, [{"id": r.get("id"), "src": r["src"], "ref": r["tgt"], "hyp": h} for r, h in zip(rows, hyps)])
    return hyps, score(hyps, [r["tgt"] for r in rows], cfg.tgt_lang)


def evaluate_candidate(cfg: PipelineConfig, method: str, smoke: Dict[str, Any], logger=None) -> Dict[str, Any]:
    """Prompt-validation metrics + judge-calibration outputs. The sealed split is NOT touched."""
    n_pv = 5 if cfg.dry_run else cfg.eval.n_prompt_validation
    n_jc = 5 if cfg.dry_run else cfg.eval.n_judge_calibration
    tok, model = load_candidate(cfg, method, smoke)
    eval_dir = os.path.join(cfg.output_dir, "eval")
    pv_rows = load_split(cfg.data_dir, "prompt_validation", n=n_pv)
    pv_hyps, pv_scores = _run_set(tok, model, cfg, pv_rows, os.path.join(eval_dir, "prompt_validation_outputs.jsonl"))
    jc_rows = load_split(cfg.data_dir, "judge_calibration", n=n_jc)
    _, jc_scores = _run_set(tok, model, cfg, jc_rows, os.path.join(eval_dir, "judge_calibration_outputs.jsonl"))
    loads_and_generates_mandarin = bool(pv_hyps) and any(han_ratio(h) >= 0.3 for h in pv_hyps)
    result = {"evaluated_utc": utc_now(), "checkpoint_loaded": True,
              "generates_mandarin": loads_and_generates_mandarin if cfg.tgt_lang == "zh" else None,
              "prompt_validation": pv_scores, "judge_calibration": jc_scores,
              "samples": [{"src": r["src"], "ref": r["tgt"], "hyp": h} for r, h in zip(pv_rows[:3], pv_hyps[:3])],
              "outputs_dir": eval_dir}
    write_json(os.path.join(eval_dir, "eval_result.json"), result)
    if logger:
        logger.info("evaluate: %s", {k: v for k, v in result.items() if k != "samples"})
    del model
    return result


def run_sealed_final_test(cfg: PipelineConfig, method: str, smoke: Dict[str, Any], manifest_path: str,
                          confirm: bool = False, logger=None) -> Dict[str, Any]:
    """The ONE-TIME final evaluation. Requires confirm=True, verifies the sealed hash first."""
    if not confirm:
        raise PermissionError("Sealed final test: call with confirm=True only once, after all iteration is done.")
    ok, problems = verify_data(cfg.data_dir)
    if not ok:
        raise RuntimeError(f"data verification failed: {problems}")
    manifest = read_json(manifest_path)
    expected = manifest["data"]["sealed"]["final_test"]["sha256"]
    actual = sha256_file(split_path(cfg.data_dir, "final_test"))
    if expected != actual:
        raise RuntimeError("sealed final_test hash does not match candidate_manifest.json; aborting")
    if manifest.get("sealed_test", {}).get("evaluated"):
        raise RuntimeError("sealed final test was already evaluated for this candidate; refusing to re-run")
    tok, model = load_candidate(cfg, method, smoke)
    rows = load_split(cfg.data_dir, "final_test", unseal=True)
    out = os.path.join(cfg.output_dir, "eval", "final_test_outputs.jsonl")
    _, metrics = _run_set(tok, model, cfg, rows, out)
    manifest["sealed_test"] = {"evaluated": True, "evaluated_utc": utc_now(), "sha256": actual,
                               "metrics": metrics, "outputs": out}
    write_json(manifest_path, manifest)
    if logger:
        logger.info("sealed final test: %s", metrics)
    return metrics

Writing tlpipe/evaluate.py


In [10]:
%%writefile tlpipe/manifest.py
"""candidate_manifest.json + report.md."""
from __future__ import annotations

import os
from typing import Any, Dict

from .config import PipelineConfig
from .utils import env_info, git_info, read_json, sha256_dir, utc_now, write_json


def write_manifest(cfg: PipelineConfig, smoke: Dict[str, Any], method: str, method_reason: str,
                   data_manifest: Dict[str, Any], train_summary: Dict[str, Any],
                   eval_result: Dict[str, Any]) -> str:
    cand_dir = train_summary["candidate_dir"]
    cfg_hash = cfg.config_hash()
    candidate_id = f"{cfg.run_name}-{cfg.direction}-{method}-{data_manifest['data_version']}-{cfg_hash[:8]}"
    manifest = {
        "candidate_id": candidate_id, "created_utc": utc_now(), "dry_run": cfg.dry_run,
        "config": cfg.to_dict(), "config_hash": cfg_hash,
        "config_path": os.path.join(cfg.output_dir, "config.json"),
        "model": {"base": cfg.model_name, "method": method, "method_reason": method_reason,
                  "candidate_dir": cand_dir, "candidate_hashes": sha256_dir(cand_dir),
                  "last_checkpoint": train_summary.get("last_checkpoint"), "loadable": eval_result.get("checkpoint_loaded")},
        "data": {"data_version": data_manifest["data_version"], "data_dir": cfg.data_dir,
                 "source": data_manifest["source"], "validation_stats": data_manifest["validation_stats"],
                 "splits": data_manifest["splits"], "sealed": data_manifest["sealed"]},
        "hardware": smoke, "environment": env_info(), "git": {**git_info(), "repo_url": cfg.repo_url},
        "training": {k: v for k, v in train_summary.items() if k != "log_history"},
        "evaluation": eval_result, "sealed_test": {"evaluated": False,
                                                  "sha256": data_manifest["sealed"]["final_test"]["sha256"]},
        "logs": {"pipeline_log": os.path.join(cfg.log_dir, "pipeline.log"),
                 "train_log": os.path.join(cfg.log_dir, "train_log.jsonl")},
    }
    path = os.path.join(cfg.output_dir, "candidate_manifest.json")
    write_json(path, manifest)
    return path


def write_report(manifest_path: str) -> str:
    m = read_json(manifest_path)
    ev, tr, hw = m["evaluation"], m["training"], m["hardware"]
    lines = [f"# Candidate report: `{m['candidate_id']}`", "",
             f"- Created: {m['created_utc']}  |  dry run: {m['dry_run']}",
             f"- Base model: `{m['model']['base']}`  |  method: **{m['model']['method']}** ({m['model']['method_reason']})",
             f"- Direction: `{m['config']['direction']}`  |  seed: {m['config']['seed']}  |  config hash: `{m['config_hash'][:12]}`",
             f"- Data version: `{m['data']['data_version']}`  |  source: {m['data']['source']['name'] or m['data']['source']['path']}",
             f"- Hardware: {hw.get('platform')} / {hw.get('gpu') or 'CPU'} ({hw.get('vram_gb')} GB), bf16={hw.get('bf16')}, bnb={hw.get('bitsandbytes')}",
             f"- Git: {m['git'].get('repo_url')} @ {m['git'].get('commit')}", "",
             "## Data", "",
             "| split | rows | sha256 |", "|---|---:|---|"]
    for name, e in {**m["data"]["splits"], **{k + " (SEALED)": v for k, v in m["data"]["sealed"].items()}}.items():
        lines.append(f"| {name} | {e['rows']} | `{e['sha256'][:16]}…` |")
    lines += ["", f"Validation stats: `{m['data']['validation_stats']}`", "",
              "## Training", "",
              f"- steps: {tr.get('global_step')}  |  train loss: {tr.get('train_loss')}  |  dev loss: {tr.get('eval_loss')}",
              f"- runtime: {tr.get('train_runtime_sec')} s  |  rows train/dev: {tr.get('rows_train')}/{tr.get('rows_dev')}",
              f"- trainable params: {tr.get('trainable_params'):,} / {tr.get('total_params'):,}  |  resumed from: {tr.get('resumed_from')}",
              "", "## Evaluation (prompt_validation, NOT the sealed test)", "",
              f"- checkpoint loaded: {ev.get('checkpoint_loaded')}  |  generates Mandarin: {ev.get('generates_mandarin')}",
              f"- prompt_validation: {ev.get('prompt_validation')}",
              f"- judge_calibration outputs: {ev.get('judge_calibration')}", "", "### Samples", ""]
    for s in ev.get("samples", []):
        lines += [f"- **src:** {s['src']}", f"  - **ref:** {s['ref']}", f"  - **hyp:** {s['hyp']}"]
    st = m.get("sealed_test", {})
    lines += ["", "## Sealed final test", "",
              f"- evaluated: {st.get('evaluated')}  |  metrics: {st.get('metrics')}", "",
              "## Reproduce", "",
              "```", f"cfg = PipelineConfig.from_json('{m['config_path']}')", "run_pipeline(cfg)", "```"]
    path = os.path.join(os.path.dirname(manifest_path), "report.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    return path

Writing tlpipe/manifest.py


In [11]:
%%writefile tlpipe/run.py
"""Orchestrator: smoke -> data -> train -> evaluate -> manifest/report."""
from __future__ import annotations

import os
from typing import Any, Dict, Iterable

from .config import PipelineConfig
from .data import prepare_data
from .evaluate import evaluate_candidate
from .hardware import select_method, smoke_test
from .manifest import write_manifest, write_report
from .train import train
from .utils import read_json, set_seed, setup_logging, write_json

STAGES = ("smoke", "data", "train", "evaluate", "manifest")


def run_pipeline(cfg: PipelineConfig, stages: Iterable[str] = STAGES) -> Dict[str, Any]:
    if cfg.dry_run and not cfg.output_dir.endswith("_dryrun"):
        cfg.output_dir = cfg.output_dir + "_dryrun"
        cfg.log_dir = os.path.join(cfg.output_dir, "logs")
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logging(cfg.log_dir)
    set_seed(cfg.seed)
    cfg.to_json(os.path.join(cfg.output_dir, "config.json"))
    logger.info("pipeline start run=%s dry_run=%s stages=%s", cfg.run_name, cfg.dry_run, list(stages))
    out: Dict[str, Any] = {"output_dir": cfg.output_dir}
    stages = set(stages)

    smoke_path = os.path.join(cfg.output_dir, "smoke_test.json")
    if "smoke" in stages or not os.path.exists(smoke_path):
        smoke = smoke_test(logger)
        method, reason = select_method(smoke, cfg.train.method)
        write_json(smoke_path, {"smoke": smoke, "method": method, "method_reason": reason})
    else:
        s = read_json(smoke_path)
        smoke, method, reason = s["smoke"], s["method"], s["method_reason"]
    logger.info("method selected: %s (%s)", method, reason)
    out.update(smoke=smoke, method=method, method_reason=reason)

    data_manifest = prepare_data(cfg, logger)
    out["data_manifest"] = data_manifest

    summary_path = os.path.join(cfg.output_dir, "train_summary.json")
    if "train" in stages:
        out["train"] = train(cfg, method, smoke, logger)
    elif os.path.exists(summary_path):
        out["train"] = read_json(summary_path)

    eval_path = os.path.join(cfg.output_dir, "eval", "eval_result.json")
    if "evaluate" in stages:
        out["evaluation"] = evaluate_candidate(cfg, method, smoke, logger)
    elif os.path.exists(eval_path):
        out["evaluation"] = read_json(eval_path)

    if "manifest" in stages and "train" in out and "evaluation" in out:
        mp = write_manifest(cfg, smoke, method, reason, data_manifest, out["train"], out["evaluation"])
        out["manifest_path"] = mp
        out["report_path"] = write_report(mp)
        logger.info("manifest -> %s | report -> %s", mp, out["report_path"])
    logger.info("pipeline done")
    return out

Writing tlpipe/run.py


In [12]:
%%writefile tests/test_pipeline.py
"""Unit tests that run without a GPU or model download:  python -m pytest -q tests"""
import json
import os
import sys
import tempfile
import unittest

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

from tlpipe.config import PipelineConfig  # noqa: E402
from tlpipe.data import (SealedSplitError, compute_data_version, finetune_train_dev,  # noqa: E402
                         han_ratio, load_split, prepare_data, split_rows, validate_rows, verify_data)
from tlpipe.hardware import select_method  # noqa: E402

try:
    import torch  # noqa: F401
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False


def _rows(n=400):
    rows = [{"src": f"Sentence number {i} about the weather.", "tgt": f"第{i}句关于天气。"} for i in range(n)]
    rows += [{"src": "", "tgt": "空"}, {"src": "dup", "tgt": "重复"}, {"src": "dup", "tgt": "重复二"},
             {"src": "same", "tgt": "same"}, {"src": "Not mandarin", "tgt": "not mandarin at all"},
             {"src": None, "tgt": "无"}, {"src": "x" * 1000, "tgt": "长"}]
    return rows


def _cfg(tmp, n=400):
    path = os.path.join(tmp, "raw.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for r in _rows(n):
            f.write(json.dumps({"english": r["src"], "chinese": r["tgt"]}, ensure_ascii=False) + "\n")
    return PipelineConfig.from_dict({
        "run_name": "t", "root_dir": tmp, "direction": "en-zh", "seed": 7,
        "dataset": {"source": "local", "path": path, "translation_column": None, "src_field": "english",
                    "tgt_field": "chinese", "max_rows": 300, "min_chars": 1, "max_chars": 200},
    })


class ConfigTests(unittest.TestCase):
    def test_roundtrip_and_hash(self):
        with tempfile.TemporaryDirectory() as tmp:
            c = PipelineConfig(run_name="r", root_dir=tmp)
            c.train.lora_r = 8
            p = c.to_json(os.path.join(tmp, "c.json"))
            c2 = PipelineConfig.from_json(p)
            self.assertEqual(c2.train.lora_r, 8)
            self.assertEqual(c.config_hash(), c2.config_hash())
            c2.seed = 1
            self.assertNotEqual(c.config_hash(), c2.config_hash())
            self.assertEqual((c.src_lang, c.tgt_lang), ("en", "zh"))

    def test_bad_direction(self):
        with self.assertRaises(ValueError):
            PipelineConfig(direction="enzh")


class MethodSelectionTests(unittest.TestCase):
    def test_auto(self):
        self.assertEqual(select_method({"cuda": False, "matmul_ok": True})[0], "lora")
        self.assertEqual(select_method({"cuda": True, "vram_gb": 15, "bitsandbytes": True, "matmul_ok": True})[0], "qlora")
        self.assertEqual(select_method({"cuda": True, "vram_gb": 15, "bitsandbytes": False, "matmul_ok": True})[0], "lora")
        self.assertEqual(select_method({"cuda": True, "vram_gb": 40, "bitsandbytes": True, "matmul_ok": True})[0], "lora")

    def test_downgrade_and_errors(self):
        self.assertEqual(select_method({"cuda": True, "vram_gb": 15, "bitsandbytes": False, "matmul_ok": True}, "qlora")[0], "lora")
        self.assertEqual(select_method({"cuda": True, "vram_gb": 15, "bitsandbytes": True, "matmul_ok": True}, "full")[0], "qlora")
        with self.assertRaises(ValueError):
            select_method({"cuda": True, "matmul_ok": True}, "nope")
        with self.assertRaises(RuntimeError):
            select_method({"cuda": True, "matmul_ok": False})


class DataTests(unittest.TestCase):
    def test_validate(self):
        cfg = PipelineConfig(direction="en-zh")
        clean, stats = validate_rows(_rows(10), cfg.dataset, "en-zh")
        self.assertEqual(stats["kept"], 11)  # 10 good + first "dup"
        self.assertEqual(stats["drop_empty"], 1)
        self.assertEqual(stats["drop_duplicate_source"], 1)
        self.assertEqual(stats["drop_identical"], 1)
        self.assertEqual(stats["drop_target_not_mandarin"], 1)
        self.assertEqual(stats["drop_not_string"], 1)
        self.assertEqual(stats["drop_length"], 1)
        self.assertEqual(han_ratio("你好 世界"), 1.0)
        self.assertEqual(han_ratio("hello"), 0.0)

    def test_split_disjoint_deterministic(self):
        cfg = PipelineConfig()
        clean, _ = validate_rows(_rows(400), cfg.dataset, "en-zh")
        a = split_rows(clean, cfg.dataset.split_ratios, 3, 300)
        b = split_rows(clean, cfg.dataset.split_ratios, 3, 300)
        self.assertEqual(a, b)
        self.assertEqual(sum(len(v) for v in a.values()), 300)
        srcs = [r["src"] for v in a.values() for r in v]
        self.assertEqual(len(srcs), len(set(srcs)))
        self.assertNotEqual(split_rows(clean, cfg.dataset.split_ratios, 4, 300), a)
        self.assertEqual(compute_data_version(a, cfg.dataset, 3), compute_data_version(b, cfg.dataset, 3))
        with self.assertRaises(ValueError):
            split_rows(clean, {"finetune": 1.0}, 3)

    def test_prepare_verify_seal_resume(self):
        with tempfile.TemporaryDirectory() as tmp:
            cfg = _cfg(tmp)
            m = prepare_data(cfg)
            self.assertTrue(m["data_version"].startswith("v-"))
            self.assertIn("final_test", m["sealed"])
            self.assertNotIn("final_test", m["splits"])
            self.assertTrue(verify_data(cfg.data_dir)[0])
            self.assertTrue(os.path.exists(os.path.join(cfg.data_dir, "sealed", "SEALED.json")))
            with self.assertRaises(SealedSplitError):
                load_split(cfg.data_dir, "final_test")
            self.assertEqual(len(load_split(cfg.data_dir, "final_test", unseal=True)), m["sealed"]["final_test"]["rows"])
            self.assertEqual(len(load_split(cfg.data_dir, "prompt_validation", n=3)), 3)
            # idempotent reuse
            self.assertEqual(prepare_data(cfg)["data_version"], m["data_version"])
            # tamper -> verify fails
            p = m["splits"]["finetune"]["path"]
            os.chmod(p, 0o644)
            with open(p, "a", encoding="utf-8") as f:
                f.write('{"src":"x","tgt":"y"}\n')
            ok, problems = verify_data(cfg.data_dir)
            self.assertFalse(ok)
            self.assertIn("finetune: sha256 mismatch", problems)
            train, dev = finetune_train_dev(cfg.data_dir, 0.1, 1, n=8)
            self.assertEqual(len(train), 8)
            self.assertGreaterEqual(len(dev), 1)
            self.assertTrue(set(r["src"] for r in train).isdisjoint(r["src"] for r in dev))


@unittest.skipUnless(HAS_TORCH, "torch not installed")
class CollatorTests(unittest.TestCase):
    def test_padding(self):
        from tlpipe.train import PadCollator
        b = PadCollator(0)([{"input_ids": [1, 2, 3], "attention_mask": [1, 1, 1], "labels": [-100, 2, 3]},
                            {"input_ids": [4], "attention_mask": [1], "labels": [4]}])
        self.assertEqual(b["input_ids"].shape, (2, 3))
        self.assertEqual(b["labels"][1].tolist(), [4, -100, -100])
        self.assertEqual(b["attention_mask"][1].tolist(), [1, 0, 0])


if __name__ == "__main__":
    unittest.main()

Writing tests/test_pipeline.py


In [13]:
%%writefile configs/default_en_zh.json
{
  "run_name": "mandarin-candidate",
  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
  "direction": "en-zh",
  "seed": 42,
  "dry_run": false,
  "root_dir": null,
  "output_dir": null,
  "data_dir": null,
  "log_dir": null,
  "repo_url": "https://github.com/<org>/<repo>",
  "dataset": {
    "source": "hf",
    "name": "Helsinki-NLP/opus-100",
    "config": "en-zh",
    "split": "train",
    "path": null,
    "translation_column": "translation",
    "src_field": "en",
    "tgt_field": "zh",
    "max_rows": 8000,
    "min_chars": 1,
    "max_chars": 400,
    "han_ratio_min": 0.3,
    "split_ratios": {
      "finetune": 0.88,
      "judge_calibration": 0.04,
      "prompt_validation": 0.04,
      "final_test": 0.04
    },
    "dev_fraction": 0.05
  },
  "train": {
    "method": "auto",
    "epochs": 1.0,
    "max_steps": -1,
    "learning_rate": 0.0002,
    "per_device_batch_size": 4,
    "grad_accum": 4,
    "max_len": 256,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "logging_steps": 10,
    "save_steps": 100,
    "save_total_limit": 3,
    "eval_steps": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "gradient_checkpointing": true,
    "resume": true,
    "dry_run_steps": 3,
    "dry_run_rows": 32
  },
  "eval": {
    "n_prompt_validation": 50,
    "n_judge_calibration": 30,
    "max_new_tokens": 128,
    "save_merged": false
  }
}

Writing configs/default_en_zh.json


In [14]:
%%writefile configs/default_zh_en.json
{
  "run_name": "zh-en-candidate",
  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
  "direction": "zh-en",
  "seed": 42,
  "dry_run": false,
  "root_dir": null,
  "output_dir": null,
  "data_dir": null,
  "log_dir": null,
  "repo_url": "https://github.com/<org>/<repo>",
  "dataset": {
    "source": "hf",
    "name": "Helsinki-NLP/opus-100",
    "config": "en-zh",
    "split": "train",
    "path": null,
    "translation_column": "translation",
    "src_field": "zh",
    "tgt_field": "en",
    "max_rows": 8000,
    "min_chars": 1,
    "max_chars": 400,
    "han_ratio_min": 0.3,
    "split_ratios": {
      "finetune": 0.88,
      "judge_calibration": 0.04,
      "prompt_validation": 0.04,
      "final_test": 0.04
    },
    "dev_fraction": 0.05
  },
  "train": {
    "method": "auto",
    "epochs": 1.0,
    "max_steps": -1,
    "learning_rate": 0.0002,
    "per_device_batch_size": 4,
    "grad_accum": 4,
    "max_len": 256,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "logging_steps": 10,
    "save_steps": 100,
    "save_total_limit": 3,
    "eval_steps": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "gradient_checkpointing": true,
    "resume": true,
    "dry_run_steps": 3,
    "dry_run_rows": 32
  },
  "eval": {
    "n_prompt_validation": 50,
    "n_judge_calibration": 30,
    "max_new_tokens": 128,
    "save_merged": false
  }
}

Writing configs/default_zh_en.json


In [15]:
%%writefile configs/dry_run_en_zh.json
{
  "run_name": "mandarin-candidate",
  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
  "direction": "en-zh",
  "seed": 42,
  "dry_run": true,
  "root_dir": null,
  "output_dir": null,
  "data_dir": null,
  "log_dir": null,
  "repo_url": "https://github.com/<org>/<repo>",
  "dataset": {
    "source": "hf",
    "name": "Helsinki-NLP/opus-100",
    "config": "en-zh",
    "split": "train",
    "path": null,
    "translation_column": "translation",
    "src_field": "en",
    "tgt_field": "zh",
    "max_rows": 600,
    "min_chars": 1,
    "max_chars": 400,
    "han_ratio_min": 0.3,
    "split_ratios": {
      "finetune": 0.88,
      "judge_calibration": 0.04,
      "prompt_validation": 0.04,
      "final_test": 0.04
    },
    "dev_fraction": 0.05
  },
  "train": {
    "method": "auto",
    "epochs": 1.0,
    "max_steps": -1,
    "learning_rate": 0.0002,
    "per_device_batch_size": 4,
    "grad_accum": 4,
    "max_len": 256,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "logging_steps": 10,
    "save_steps": 100,
    "save_total_limit": 3,
    "eval_steps": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "gradient_checkpointing": true,
    "resume": true,
    "dry_run_steps": 3,
    "dry_run_rows": 32
  },
  "eval": {
    "n_prompt_validation": 50,
    "n_judge_calibration": 30,
    "max_new_tokens": 128,
    "save_merged": false
  }
}

Writing configs/dry_run_en_zh.json


In [16]:
%%writefile README.md
# tlpipe — reproducible translation fine-tuning pipeline

Configurable, versioned fine-tuning of a causal LM for translation (default: English → Mandarin,
`Qwen/Qwen2.5-0.5B-Instruct`). Runs on Colab, Kaggle or locally.

## What it does
1. **Hardware smoke test** → picks a feasible method (`lora` / `qlora` / `full`, or honours your request).
2. **Data**: load (HF hub or local jsonl/csv/tsv) → validate (types, lengths, duplicates, Han-script check) →
   seeded split → write JSONL → sha256 every file → `data_manifest.json` with a `data_version`.
   Splits: `finetune`, `judge_calibration`, `prompt_validation`, and a **sealed** `final_test`
   (stored under `data/<dir>/sealed/`, read-only, refused by `load_split()` unless `unseal=True`).
3. **Train**: HF `Trainer` + PEFT; checkpoints every `save_steps`; auto-**resume** from the last checkpoint;
   `dry_run=True` runs 3 steps on 32 rows through every stage; logs in `logs/pipeline.log` and `logs/train_log.jsonl`.
4. **Evaluate**: fresh load of the candidate (base + adapter), greedy generation on `prompt_validation`
   (BLEU/chrF via sacrebleu, Mandarin-output check) and outputs on `judge_calibration`.
5. **Manifest**: `candidate_manifest.json` (config + hash, data version + hashes, candidate weight hashes,
   hardware, environment, git, training and eval summaries) and `report.md`.
6. **Sealed final test**: `run_sealed_final_test(..., confirm=True)` — verifies the sealed hash, runs once, records
   results in the manifest and refuses to run twice.

## Quick start
```bash
pip install -r requirements.txt
python -m unittest tests.test_pipeline
python - <<'PY'
from tlpipe.config import PipelineConfig
from tlpipe.run import run_pipeline
cfg = PipelineConfig.from_json("configs/dry_run_en_zh.json"); run_pipeline(cfg)   # dry run
cfg = PipelineConfig.from_json("configs/default_en_zh.json"); run_pipeline(cfg)   # real run
PY
```
Or open `tlpipe_colab.ipynb` in Colab/Kaggle — it writes this repo layout, runs the tests, the dry run and the full run.

## Layout
```
tlpipe/         config.py hardware.py data.py train.py evaluate.py manifest.py run.py utils.py
configs/        default_en_zh.json dry_run_en_zh.json default_zh_en.json
tests/          test_pipeline.py
tlpipe_colab.ipynb
```
Run outputs land in `<root_dir>/<run_name>/`: `config.json`, `smoke_test.json`, `checkpoints/`, `candidate/`,
`eval/`, `logs/`, `train_summary.json`, `candidate_manifest.json`, `report.md`.

Writing README.md


In [17]:
%%writefile requirements.txt
torch>=2.2
transformers>=4.44
peft>=0.12
datasets>=2.19
accelerate>=0.30
bitsandbytes>=0.43
sacrebleu>=2.4
psutil

Writing requirements.txt


In [18]:
# Unit tests: config round-trip/hash, method selection, validation, disjoint deterministic splits,
# data versioning + sha256 verification, sealed-split refusal, tamper detection, collator padding.
!python -m unittest -v tests.test_pipeline

test_pipeline (unittest.loader._FailedTest.test_pipeline) ... ERROR

ERROR: test_pipeline (unittest.loader._FailedTest.test_pipeline)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_pipeline
Traceback (most recent call last):
  File "/usr/lib/python3.13/unittest/loader.py", line 141, in loadTestsFromName
    module = __import__(module_name)
ModuleNotFoundError: No module named 'tests.test_pipeline'


----------------------------------------------------------------------
Ran 1 test in 0.000s

FAILED (errors=1)


### 2b. Environment compatibility fix (run once per session)
Colab preinstalls `torchao 0.10`, which recent `peft` rejects outright even though LoRA/QLoRA never use it, and some `transformers` builds have dropped `TrainingArguments(warmup_ratio=...)`. This cell neutralises both. It is idempotent — **re-run it after any runtime restart.**

In [19]:
# =========================================================================================
# FIX CELL — run once per Colab session, after section 2, before section 5.
# Safe to re-run. Re-run it after every runtime restart.
#   (a) neutralises PEFT's torchao version guard (Colab ships torchao 0.10; peft wants >=0.16;
#       LoRA/QLoRA never touch torchao, so the guard is pure noise for us)
#   (b) makes TrainingArguments construction tolerant of transformers version drift
#       (warmup_ratio / eval_strategy renames and removals)
# =========================================================================================
import sys, inspect, importlib

# ---------- (a) torchao guard -------------------------------------------------------------
import peft, peft.import_utils

def _no_torchao():
    return False

_patched_in = []
peft.import_utils.is_torchao_available = _no_torchao
for name, mod in list(sys.modules.items()):
    if name.split(".")[0] in ("peft", "transformers") and getattr(mod, "is_torchao_available", None) is not None:
        mod.is_torchao_available = _no_torchao
        _patched_in.append(name)
print("torchao guard disabled in:", _patched_in)
from peft.import_utils import is_torchao_available
assert is_torchao_available() is False, "torchao guard still active"

# ---------- (b) TrainingArguments compatibility ------------------------------------------
NEW = '''
def _supported_params(cls):
    # parameter names accepted by cls.__init__, and whether it takes **kwargs
    params = inspect.signature(cls.__init__).parameters
    var_kw = any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())
    return set(params) - {"self"}, var_kw


def _estimate_total_steps(cfg, n_rows):
    # optimizer steps the run will take; used when only warmup_steps is supported
    max_steps = cfg.train.dry_run_steps if cfg.dry_run else cfg.train.max_steps
    if max_steps and max_steps > 0:
        return int(max_steps)
    if not n_rows:
        return 0
    per_epoch = math.ceil(n_rows / max(1, cfg.train.per_device_batch_size * cfg.train.grad_accum))
    return max(1, int(per_epoch * cfg.train.epochs))


def _training_args(cfg, method, smoke, ckpt_dir, n_rows=None, logger=None):
    # build TrainingArguments, tolerating renamed/removed arguments across transformers versions
    from transformers import TrainingArguments
    t, cuda = cfg.train, smoke.get("cuda")
    supported, var_kw = _supported_params(TrainingArguments)
    max_steps = t.dry_run_steps if cfg.dry_run else t.max_steps
    kw = dict(
        output_dir=ckpt_dir, seed=cfg.seed, data_seed=cfg.seed,
        per_device_train_batch_size=t.per_device_batch_size, per_device_eval_batch_size=t.per_device_batch_size,
        gradient_accumulation_steps=t.grad_accum, learning_rate=t.learning_rate,
        num_train_epochs=t.epochs, max_steps=max_steps,
        weight_decay=t.weight_decay, lr_scheduler_type="cosine",
        logging_steps=1 if cfg.dry_run else t.logging_steps, logging_first_step=True,
        save_steps=1 if cfg.dry_run else t.save_steps, save_total_limit=t.save_total_limit,
        eval_steps=1 if cfg.dry_run else t.eval_steps, save_strategy="steps",
        bf16=bool(cuda and smoke.get("bf16")), fp16=bool(cuda and not smoke.get("bf16")),
        report_to="none", remove_unused_columns=False, dataloader_num_workers=0,
        optim="paged_adamw_8bit" if method == "qlora" else "adamw_torch",
        gradient_checkpointing=False,
    )
    for alias in ("eval_strategy", "evaluation_strategy"):
        if var_kw or alias in supported:
            kw[alias] = "steps"
            break
    if var_kw or "warmup_ratio" in supported:
        kw["warmup_ratio"] = t.warmup_ratio
    elif "warmup_steps" in supported:
        total = _estimate_total_steps(cfg, n_rows)
        kw["warmup_steps"] = max(0, int(round(total * t.warmup_ratio)))
        if logger:
            logger.info("warmup_ratio unsupported -> warmup_steps=%s (of ~%s total)", kw["warmup_steps"], total)
    if not var_kw:
        dropped = sorted(k for k in kw if k not in supported)
        for k in dropped:
            kw.pop(k)
        if dropped:
            msg = "TrainingArguments does not accept %s; dropped, using its defaults." % dropped
            (logger.warning if logger else print)(msg)
    return TrainingArguments(**kw)
'''

src = open("tlpipe/train.py", encoding="utf-8").read()
if "_supported_params" in src:
    print("train.py already patched")
else:
    start = src.index("def _training_args(")
    end = src.index("# ----------------------------------------------------------------------------- run")
    src = src[:start] + NEW.strip("\n") + "\n\n\n" + src[end:]
    src = src.replace("_training_args(cfg, method, smoke, ckpt_dir)",
                      "_training_args(cfg, method, smoke, ckpt_dir, n_rows=len(train_ds), logger=logger)")
    if "\nimport math" not in src:
        src = src.replace("import inspect\nimport json", "import inspect\nimport json\nimport math")
    open("tlpipe/train.py", "w", encoding="utf-8").write(src)
    print("train.py patched")

for m in [m for m in list(sys.modules) if m.startswith("tlpipe")]:
    del sys.modules[m]
from tlpipe.run import run_pipeline
from tlpipe.train import _supported_params
import transformers
sup, var_kw = _supported_params(transformers.TrainingArguments)
print("transformers", transformers.__version__, "| peft", peft.__version__)
print("warmup_ratio:", "warmup_ratio" in sup, "| warmup_steps:", "warmup_steps" in sup,
      "| eval_strategy:", "eval_strategy" in sup, "| evaluation_strategy:", "evaluation_strategy" in sup,
      "| **kwargs:", var_kw)
print("READY — now re-run section 5 (dry run).")

torchao guard disabled in: ['transformers.utils.import_utils', 'transformers.utils', 'transformers.utils.quantization_config', 'transformers.quantizers.quantizer_torchao', 'peft.import_utils', 'peft.utils.quantization_utils', 'peft.tuners.lora.torchao']
train.py already patched
transformers 5.16.1 | peft 0.20.0
warmup_ratio: False | warmup_steps: True | eval_strategy: True | evaluation_strategy: False | **kwargs: False
READY — now re-run section 5 (dry run).


## 3. Configuration
Everything is in one JSON file. Edit the file or override attributes below. `train.method="auto"` lets the
smoke test choose; set `"lora"`, `"qlora"` or `"full"` to force (infeasible choices are downgraded with a logged reason).

In [20]:
import os, json, importlib, sys
sys.path.insert(0, os.getcwd())
import tlpipe.config, tlpipe.run
for m in list(sys.modules):
    if m.startswith("tlpipe"):
        importlib.reload(sys.modules[m])
from tlpipe.config import PipelineConfig
from tlpipe.run import run_pipeline

CONFIG_PATH = "configs/default_en_zh.json"
cfg = PipelineConfig.from_json(CONFIG_PATH)

# ---- overrides (all optional) -----------------------------------------------------------
cfg.repo_url    = "https://github.com/<org>/<repo>"        # <-- put the exact GitHub link here
cfg.run_name    = "mandarin-candidate"
cfg.model_name  = "Qwen/Qwen2.5-0.5B-Instruct"             # any chat-templated causal LM
cfg.direction   = "en-zh"                                  # "<src>-<tgt>", e.g. "zh-en"
cfg.seed        = 42
cfg.train.method = "auto"                                  # auto | lora | qlora | full
cfg.dataset.max_rows = 8000                                # rows after validation, before splitting
cfg.train.epochs, cfg.train.max_steps = 1.0, -1
cfg.train.per_device_batch_size, cfg.train.grad_accum = 4, 4
cfg.train.save_steps = cfg.train.eval_steps = 100
if os.environ.get("TLPIPE_ROOT"):
    cfg.root_dir = os.environ["TLPIPE_ROOT"]; cfg.output_dir = cfg.data_dir = cfg.log_dir = None; cfg.__post_init__()
# -----------------------------------------------------------------------------------------
print(json.dumps(cfg.to_dict(), indent=2, ensure_ascii=False))
print("config hash:", cfg.config_hash())

{
  "run_name": "mandarin-candidate",
  "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
  "direction": "en-zh",
  "seed": 42,
  "dry_run": false,
  "root_dir": "/content/tlpipe_runs",
  "output_dir": "/content/tlpipe_runs/mandarin-candidate",
  "data_dir": "/content/tlpipe_runs/data/en-zh",
  "log_dir": "/content/tlpipe_runs/mandarin-candidate/logs",
  "repo_url": "https://github.com/<org>/<repo>",
  "dataset": {
    "source": "hf",
    "name": "Helsinki-NLP/opus-100",
    "config": "en-zh",
    "split": "train",
    "path": null,
    "translation_column": "translation",
    "src_field": "en",
    "tgt_field": "zh",
    "max_rows": 8000,
    "min_chars": 1,
    "max_chars": 400,
    "han_ratio_min": 0.3,
    "split_ratios": {
      "finetune": 0.88,
      "judge_calibration": 0.04,
      "prompt_validation": 0.04,
      "final_test": 0.04
    },
    "dev_fraction": 0.05
  },
  "train": {
    "method": "auto",
    "epochs": 1.0,
    "max_steps": -1,
    "learning_rate": 0.0002,
    "per_dev

## 4. Hardware smoke test → method selection

In [21]:
from tlpipe.hardware import smoke_test, select_method
smoke = smoke_test()
method, reason = select_method(smoke, cfg.train.method)
print(json.dumps(smoke, indent=2)); print("\n=> method:", method, "|", reason)

{
  "platform": "colab",
  "cuda": true,
  "gpu": "Tesla T4",
  "vram_gb": 15.6,
  "bf16": true,
  "bitsandbytes": true,
  "matmul_ok": true,
  "torch": "2.11.0+cu128",
  "free_disk_gb": 201.9,
  "ram_gb": 13.6,
  "errors": []
}

=> method: qlora | GPU with 15.6 GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen


## 5. Dry run
Runs *every* stage (data → 3 training steps on 32 rows → checkpoint → fresh load → generation → manifest)
into `<run>_dryrun/`. Use it to validate the config before spending GPU time.

In [22]:
import copy
dry = copy.deepcopy(cfg); dry.dry_run = True
dry_out = run_pipeline(dry)
print("dry-run manifest:", dry_out["manifest_path"])
print(json.dumps(dry_out["evaluation"]["samples"], indent=2, ensure_ascii=False))

2026-09-11 04:05:32,742 INFO pipeline start run=mandarin-candidate dry_run=True stages=['smoke', 'data', 'train', 'evaluate', 'manifest']
2026-09-11 04:05:32,748 INFO smoke_test: {'platform': 'colab', 'cuda': True, 'gpu': 'Tesla T4', 'vram_gb': 15.6, 'bf16': True, 'bitsandbytes': True, 'matmul_ok': True, 'torch': '2.11.0+cu128', 'free_disk_gb': 201.9, 'ram_gb': 13.6, 'errors': []}
2026-09-11 04:05:32,751 INFO method selected: qlora (GPU with 15.6 GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen)


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-zh/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  355kB            

en-zh/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-zh/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  143MB            

...

Step,Training Loss,Validation Loss
1,3.119277,2.553955
2,2.539393,2.403916
3,2.251545,2.374019


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training Loss,Validation Loss,Step
2.251545,2.374019,3


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-09-11 04:06:40,797 INFO evaluate: {'evaluated_utc': '2026-09-11T04:06:40Z', 'checkpoint_loaded': True, 'generates_mandarin': True, 'prompt_validation': {'bleu': 20.57, 'chrf': 24.06, 'n': 5, 'mandarin_output_rate': 1.0}, 'judge_calibration': {'bleu': 2.21, 'chrf': 3.47, 'n': 5, 'mandarin_output_rate': 0.8}, 'outputs_dir': '/content/tlpipe_runs/mandarin-candidate_dryrun/eval'}
2026-09-11 04:06:41,010 INFO manifest -> /content/tlpipe_runs/mandarin-candidate_dryrun/candidate_manifest.json | report -> /content/tlpipe_runs/mandarin-candidate_dryrun/report.md
2026-09-11 04:06:41,011 INFO pipeline done
dry-run manifest: /content/tlpipe_runs/mandarin-candidate_dryrun/candidate_manifest.json
[
  {
    "src": "Although a vision for long-term change is useful, development plans and projects must have a much shorter time-frame.",
    "ref": "24. 虽然应了解长期改变的情况,但发展计划和项目必须具有较短的时限。",
    "hyp": "虽然长期目标规划是有用的，但项目和计划必须有更短的时间框架。"
  },
  {
    "src": "- Tovuz customs post",
    "ref": "- Tovuz海关检查站",


## 6. Full run (checkpoints + resume)
Checkpoints go to `<output_dir>/checkpoints/checkpoint-N`. If the runtime disconnects, **re-run this same cell**:
`train.resume=True` picks up the last checkpoint automatically (optimizer, scheduler and RNG state included).

In [23]:
out = run_pipeline(cfg)
print("output dir:", out["output_dir"])
print("train:", {k: v for k, v in out["train"].items() if k in ("method", "global_step", "train_loss", "eval_loss", "train_runtime_sec", "resumed_from")})
print("prompt_validation:", out["evaluation"]["prompt_validation"])

2026-09-11 04:06:41,022 INFO pipeline start run=mandarin-candidate dry_run=False stages=['smoke', 'data', 'train', 'evaluate', 'manifest']
2026-09-11 04:06:41,026 INFO smoke_test: {'platform': 'colab', 'cuda': True, 'gpu': 'Tesla T4', 'vram_gb': 15.6, 'bf16': True, 'bitsandbytes': True, 'matmul_ok': True, 'torch': '2.11.0+cu128', 'free_disk_gb': 200.4, 'ram_gb': 13.6, 'errors': []}
2026-09-11 04:06:41,027 INFO method selected: qlora (GPU with 15.6 GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen)
2026-09-11 04:06:41,034 INFO data: reusing verified v-70c5d28defa0 at /content/tlpipe_runs/data/en-zh
2026-09-11 04:06:46,086 INFO train: method=qlora rows train=6688 dev=352 dry_run=False


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
2026-09-11 04:06:48,120 INFO TrainingArguments: warmup_ratio unsupported -> warmup_steps=13 (of ~418 total)
2026-09-11 04:06:48,162 INFO train: resume_from=None


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,2.292113,2.388685
200,2.202901,2.314127
300,2.205122,2.269523
400,2.225748,2.257520
418,2.192605,2.257275


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training Loss,Validation Loss,Step
2.192605,2.257275,418


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-09-11 04:34:57,294 INFO evaluate: {'evaluated_utc': '2026-09-11T04:34:57Z', 'checkpoint_loaded': True, 'generates_mandarin': True, 'prompt_validation': {'bleu': 30.87, 'chrf': 28.7, 'n': 50, 'mandarin_output_rate': 1.0}, 'judge_calibration': {'bleu': 14.95, 'chrf': 16.77, 'n': 30, 'mandarin_output_rate': 1.0}, 'outputs_dir': '/content/tlpipe_runs/mandarin-candidate/eval'}
2026-09-11 04:34:57,487 INFO manifest -> /content/tlpipe_runs/mandarin-candidate/candidate_manifest.json | report -> /content/tlpipe_runs/mandarin-candidate/report.md
2026-09-11 04:34:57,488 INFO pipeline done
output dir: /content/tlpipe_runs/mandarin-candidate
train: {'method': 'qlora', 'resumed_from': None, 'global_step': 418, 'train_loss': 2.3076913493672055, 'train_runtime_sec': 1601.7, 'eval_loss': 2.257274627685547}
prompt_validation: {'bleu': 30.87, 'chrf': 28.7, 'n': 50, 'mandarin_output_rate': 1.0}


In [24]:
# Inspect logs and checkpoints
!ls -la {out["output_dir"]} {out["output_dir"]}/checkpoints {out["output_dir"]}/candidate
!tail -n 5 {out["output_dir"]}/logs/train_log.jsonl
!tail -n 8 {out["output_dir"]}/logs/pipeline.log

/content/tlpipe_runs/mandarin-candidate:
total 48
drwxr-xr-x 6 root root 4096 Sep 11 04:34 .
drwxr-xr-x 5 root root 4096 Sep 11 04:06 ..
drwxr-xr-x 2 root root 4096 Sep 11 04:33 candidate
-rw-r--r-- 1 root root 7665 Sep 11 04:34 candidate_manifest.json
drwxr-xr-x 5 root root 4096 Sep 11 04:33 checkpoints
-rw-r--r-- 1 root root 1664 Sep 11 04:06 config.json
drwxr-xr-x 2 root root 4096 Sep 11 04:34 eval
drwxr-xr-x 2 root root 4096 Sep 11 04:06 logs
-rw-r--r-- 1 root root 2616 Sep 11 04:34 report.md
-rw-r--r-- 1 root root  379 Sep 11 04:06 smoke_test.json
-rw-r--r-- 1 root root 1589 Sep 11 04:33 train_summary.json

/content/tlpipe_runs/mandarin-candidate/candidate:
total 45604
drwxr-xr-x 2 root root     4096 Sep 11 04:33 .
drwxr-xr-x 6 root root     4096 Sep 11 04:34 ..
-rw-r--r-- 1 root root     1159 Sep 11 04:33 adapter_config.json
-rw------- 1 root root 35237104 Sep 11 04:33 adapter_model.safetensors
-rw-r--r-- 1 root root     2507 Sep 11 04:33 chat_template.jinja
-rw-r--r-- 1 root roo

## 7. Load the candidate checkpoint and translate
Independent proof that the saved candidate loads (base + adapter from disk) and produces Mandarin.

In [25]:
from tlpipe.evaluate import load_candidate, translate
from tlpipe.data import han_ratio
tok, model = load_candidate(cfg, out["method"], out["smoke"])
sentences = ["The train to Shanghai leaves at seven tomorrow morning.",
             "Please send me the quarterly report before the meeting.",
             "Machine learning models need clean, well-documented data."]
for s, h in zip(sentences, translate(tok, model, cfg, sentences)):
    print(f"{s}\n  -> {h}   (han ratio {han_ratio(h):.2f})")
assert all(han_ratio(h) >= 0.3 for h in translate(tok, model, cfg, sentences)), "candidate did not produce Mandarin"
del model; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The train to Shanghai leaves at seven tomorrow morning.
  -> 上海开往上海的火车明天早上七点出发   (han ratio 1.00)
Please send me the quarterly report before the meeting.
  -> 请在会议前寄出季度报告。   (han ratio 0.92)
Machine learning models need clean, well-documented data.
  -> 机器学习模型需要干净、有详细记录的数据。   (han ratio 0.90)


## 8. Manifest and report

In [26]:
from tlpipe.utils import read_json
m = read_json(out["manifest_path"])
print(json.dumps({k: m[k] for k in ("candidate_id", "config_hash", "model", "data", "sealed_test", "git")}, indent=2, ensure_ascii=False)[:4000])
from IPython.display import Markdown, display
display(Markdown(open(out["report_path"], encoding="utf-8").read()))

{
  "candidate_id": "mandarin-candidate-en-zh-qlora-v-70c5d28defa0-de8409a0",
  "config_hash": "de8409a0e7499bc2dc16147e944e8d103ecaf41e95d2c5d75d5ce38b94ac868f",
  "model": {
    "base": "Qwen/Qwen2.5-0.5B-Instruct",
    "method": "qlora",
    "method_reason": "GPU with 15.6 GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen",
    "candidate_dir": "/content/tlpipe_runs/mandarin-candidate/candidate",
    "candidate_hashes": {
      "adapter_config.json": "f712d7108482ee37d61017827a687856e3b7583c50992ba7abb2585a6187a9ea",
      "adapter_model.safetensors": "5def8be22a85f60cdf44002703640bb1a7509a3b424a8022676ab724b9c8c91e",
      "tokenizer.json": "3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8",
      "tokenizer_config.json": "04b1682c59acbd057f4c9072297faa73d56fc9de053094c659cdb4c464f58f86",
      "training_args.bin": "fa4b81c1ac979542c44fcd41b5b9033788dce163014412da7f1b8f08aca81fae"
    },
    "last_checkpoint": "/content/tlpipe_runs/mandarin-candidate/checkpoint

# Candidate report: `mandarin-candidate-en-zh-qlora-v-70c5d28defa0-de8409a0`

- Created: 2026-09-11T04:34:57Z  |  dry run: False
- Base model: `Qwen/Qwen2.5-0.5B-Instruct`  |  method: **qlora** (GPU with 15.6 GB VRAM and working bitsandbytes: QLoRA (4-bit) chosen)
- Direction: `en-zh`  |  seed: 42  |  config hash: `de8409a0e749`
- Data version: `v-70c5d28defa0`  |  source: Helsinki-NLP/opus-100
- Hardware: colab / Tesla T4 (15.6 GB), bf16=True, bnb=True
- Git: https://github.com/<org>/<repo> @ None

## Data

| split | rows | sha256 |
|---|---:|---|
| finetune | 7040 | `67b5283a61588baf…` |
| judge_calibration | 320 | `3f43e4005329aa63…` |
| prompt_validation | 320 | `9e87633128a4fab8…` |
| final_test (SEALED) | 320 | `9c8f51f9de5f2bac…` |

Validation stats: `{'total': 16000, 'kept': 14544, 'drop_target_not_mandarin': 535, 'drop_identical': 99, 'drop_duplicate_source': 479, 'drop_length': 343}`

## Training

- steps: 418  |  train loss: 2.3076913493672055  |  dev loss: 2.257274627685547
- runtime: 1601.7 s  |  rows train/dev: 6688/352
- trainable params: 8,798,208 / 323,917,696  |  resumed from: None

## Evaluation (prompt_validation, NOT the sealed test)

- checkpoint loaded: True  |  generates Mandarin: True
- prompt_validation: {'bleu': 30.87, 'chrf': 28.7, 'n': 50, 'mandarin_output_rate': 1.0}
- judge_calibration outputs: {'bleu': 14.95, 'chrf': 16.77, 'n': 30, 'mandarin_output_rate': 1.0}

### Samples

- **src:** Although a vision for long-term change is useful, development plans and projects must have a much shorter time-frame.
  - **ref:** 24. 虽然应了解长期改变的情况,但发展计划和项目必须具有较短的时限。
  - **hyp:** 虽然长期改变的愿景是很有用的，但发展计划和项目必须有更短的时间框架。
- **src:** - Tovuz customs post
  - **ref:** - Tovuz海关检查站
  - **hyp:** - 通关办事处
- **src:** Delegations are urged to contact the secretariat of the Sixth Committee (Ms. Marianne Sooksatan, tel. 1 (212) 963-5378; fax 1 (212) 963-1963) if they wish to be inscribed on the list.
  - **ref:** 希望登记发言的代表团请接洽第六委员会秘书处（Marianne Sooksatan女士（电话：1（212）963-5378；传真：1（212）963-1963））。
  - **hyp:** 代表团要求秘书处（女士Marianne Sooksatan，电话：1(212) 963-5378；传真：1(212) 963-1963）如果想列入名单，请与秘书处联系。

## Sealed final test

- evaluated: False  |  metrics: None

## Reproduce

```
cfg = PipelineConfig.from_json('/content/tlpipe_runs/mandarin-candidate/config.json')
run_pipeline(cfg)
```


## 9. Package the submission
Creates `submission_<candidate_id>.zip` with code, configs, tests, manifests, logs, report, eval outputs and the
loadable candidate (adapter). Push the code to GitHub and paste the exact link into `cfg.repo_url` (section 3) so it
lands in the manifest; the zip / Drive path is the candidate download path.

In [27]:
import shutil, glob
cid = m["candidate_id"]
stage = f"/tmp/submission_{cid}"; shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for d in ("tlpipe", "tests", "configs"):
    shutil.copytree(d, os.path.join(stage, d), ignore=shutil.ignore_patterns("__pycache__"))
for f in ("README.md", "requirements.txt"):
    shutil.copy(f, stage)
for f in glob.glob("*.ipynb"):
    shutil.copy(f, stage)
run_dir = out["output_dir"]; os.makedirs(os.path.join(stage, "run"), exist_ok=True)
for item in ("candidate", "eval", "logs", "candidate_manifest.json", "report.md", "config.json", "smoke_test.json", "train_summary.json"):
    src = os.path.join(run_dir, item)
    if os.path.isdir(src): shutil.copytree(src, os.path.join(stage, "run", item))
    elif os.path.exists(src): shutil.copy(src, os.path.join(stage, "run", item))
shutil.copy(os.path.join(cfg.data_dir, "data_manifest.json"), os.path.join(stage, "run", "data_manifest.json"))
zip_path = shutil.make_archive(f"submission_{cid}", "zip", stage)
print("submission zip:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
if os.environ.get("TLPIPE_ROOT", "").startswith("/content/drive"):
    shutil.copy(zip_path, os.environ["TLPIPE_ROOT"]); print("copied to Drive:", os.environ["TLPIPE_ROOT"])

submission zip: /content/submission_mandarin-candidate-en-zh-qlora-v-70c5d28defa0-de8409a0.zip (34.8 MB)


In [28]:
# Push code to GitHub (fill in your repo; run once). The notebook + code are the repo, run outputs stay out via .gitignore.
# !git init -q && git add tlpipe tests configs README.md requirements.txt *.ipynb
# !git -c user.name="you" -c user.email="you@example.com" commit -qm "tlpipe: reproducible translation fine-tuning pipeline"
# !git branch -M main && git remote add origin https://github.com/<org>/<repo>.git && git push -u origin main

## 10. Sealed final test — run **once**, at the very end
Verifies the sealed file's sha256 against `candidate_manifest.json`, evaluates, writes the metrics into the manifest,
and refuses to run a second time for the same candidate.

In [29]:
RUN_FINAL_TEST = False   # flip to True only when iteration is finished
if RUN_FINAL_TEST:
    from tlpipe.evaluate import run_sealed_final_test
    from tlpipe.manifest import write_report
    metrics = run_sealed_final_test(cfg, out["method"], out["smoke"], out["manifest_path"], confirm=True)
    print("FINAL TEST:", metrics); write_report(out["manifest_path"])